# Fields on grids: waves, diffusion, potentials and particle-mesh gravity

Beyond particles, physim solves partial differential equations on regular grids in 1 to 3 dimensions with Dirichlet,
Neumann or periodic boundaries: the wave equation (leapfrog), the heat equation (explicit or Crank-Nicolson) and
Poisson's equation (FFT or conjugate gradients). The same machinery couples back to particles as **particle-mesh
gravity**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import physim as ps

## A drum

A square membrane clamped at its edges (`boundary="dirichlet"`), struck off-centre. Leapfrog is stable for
$c\,\Delta t/h \le 1/\sqrt 2$ in 2D; being symplectic, its energy error stays small and bounded (it oscillates
at $O(\Delta t^2)$ instead of drifting).

In [ ]:
n = 101
drum = ps.WaveEquation([n, n], spacing=1 / (n - 1), c=1.0, boundary="dirichlet")
X, Y = np.meshgrid(*drum.axes, indexing="ij")
drum.u = np.exp(-((X - 0.3) ** 2 + (Y - 0.4) ** 2) / 0.003)
e0 = drum.energy()
dt = 0.5 * drum.max_stable_dt
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, t in zip(axes, [0.0, 0.15, 0.4, 0.8]):
    drum.step(dt, int(round((t - drum.t) / dt)))
    ax.imshow(drum.u.T, origin="lower", extent=[0, 1, 0, 1], cmap="RdBu_r", vmin=-0.3, vmax=0.3)
    ax.set_title(f"t = {drum.t:.2f}"); ax.set_xticks([]); ax.set_yticks([])
print(f"energy change after {drum.t:.2f}: {drum.energy() / e0 - 1:.1e}")

## Diffusion: explicit versus Crank-Nicolson

The explicit scheme is only stable for $D\,\Delta t/h^2 \le 1/2$ in 1D. Crank-Nicolson is stable for any step and second
order in time, so it can take steps 100 times larger here and still match the exact spreading Gaussian
$u = (4\pi D t)^{-1/2} e^{-x^2/4Dt}$ (shifted to start at $t_0$).

In [ ]:
n, D, t0, t1 = 401, 1.0, 1e-3, 0.02
x = np.linspace(-1, 1, n)
exact = lambda t: np.exp(-x**2 / (4 * D * t)) / np.sqrt(4 * np.pi * D * t)
fig, ax = plt.subplots(figsize=(7, 3.5))
for k, (method, factor) in enumerate([("explicit", 0.9), ("crank_nicolson", 100)]):
    rod = ps.HeatEquation([n], x[1] - x[0], diffusivity=D, boundary="neumann", method=method)
    rod.u = exact(t0)
    dt = factor * (x[1] - x[0]) ** 2 / (2 * D)
    steps = int(round((t1 - t0) / dt))
    rod.step((t1 - t0) / steps, steps)
    err = np.abs(rod.u - exact(t1)).max() / exact(t1).max()
    ax.plot(x, rod.u, color=ps.plot.color(k), lw=2 - k, label=f"{method}: {steps} steps, error {err:.1e}")
ax.plot(x, exact(t1), "k:", lw=1, label="exact")
ax.set_xlim(-0.6, 0.6); ax.set_xlabel("x"); ax.set_ylabel("u"); ps.plot.tidy(ax);

## Electrostatics: a dipole in a grounded box

`solve_poisson` with Dirichlet boundaries (the box walls held at $\varphi = 0$) gives the potential of two opposite
point charges, $\nabla^2\varphi = -\rho/\varepsilon_0$.

In [ ]:
n = 129
h = 1 / (n - 1)
rho = np.zeros((n, n))
rho[48, 64], rho[80, 64] = 1 / h**2, -1 / h**2        # unit charges per unit length
phi = ps.solve_poisson(-rho, h, boundary="dirichlet")
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.contour(np.linspace(0, 1, n), np.linspace(0, 1, n), phi.T, levels=np.linspace(-0.6, 0.6, 25), cmap="RdBu_r")
ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title("equipotentials");

## Particle-mesh gravity: the collapse of a cold sphere

`ParticleMesh` spreads particle masses onto a grid, solves for the field with FFTs (zero-padded, so the boundaries are
open) and interpolates the forces back. Its cost does not depend on how particles are arranged, which suits smooth,
large systems. A uniform sphere released from rest collapses homologously: every mass shell follows the cycloid
$r = r_0(1 + \cos\eta)/2$, $t = t_\text{ff}(\eta + \sin\eta)/\pi$, reaching the centre at the free-fall time
$t_\text{ff} = \sqrt{3\pi/32G\rho}$. Below, 20 000 particles on a $32^3$ grid follow it until the grid can no longer
resolve the shrinking sphere.

In [ ]:
rng = np.random.default_rng(1)
n, R, M = 20_000, 1.0, 1.0
r = R * rng.uniform(0, 1, n) ** (1 / 3)
z = rng.uniform(-1, 1, n); az = rng.uniform(0, 2 * np.pi, n); s = np.sqrt(1 - z**2)
w = ps.World()
for p in np.stack([r * s * np.cos(az), r * s * np.sin(az), r * z], axis=1):
    w.add_particle(p, mass=M / n)
w.add_force(ps.ParticleMesh(box_size=2.6, cells=32))
t_ff = np.sqrt(3 * np.pi / (32 * M / (4 / 3 * np.pi * R**3)))
tr = w.run(t_ff / 100, 80, record_every=4, energies=False)

eta = np.linspace(0, np.pi, 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot((eta + np.sin(eta)) / np.pi, (1 + np.cos(eta)) / 2, "k:", lw=1.5, label="cycloid (exact)")
radii = np.sort(np.linalg.norm(tr.pos, axis=2), axis=1)
for k, f in enumerate([0.1, 0.5, 0.9]):
    ax.plot(tr.t / t_ff, radii[:, int(f * n)] / radii[0, int(f * n)], "o", ms=4, color=ps.plot.color(k),
            label=f"{f:.0%} mass radius")
ax.set_xlabel(r"$t / t_\mathrm{ff}$"); ax.set_ylabel("$r / r_0$"); ax.set_xlim(0, 1); ps.plot.tidy(ax)
half = np.argmin(np.abs(tr.t / t_ff - 0.5))
print(f"at t = 0.5 t_ff the half-mass radius is {radii[half, n // 2] / radii[0, n // 2]:.3f} r0 (cycloid: 0.836)")